In [3]:
import pandas as pd
import glob
import os

path = "MachineLearningCVE/*.csv" 
files = glob.glob(path) 
print(f"Found {len(files)} files:")

Found 8 files:


In [4]:
for f in files:
    print(" -", os.path.basename(f))

df = pd.concat([pd.read_csv(f, encoding='utf-8', low_memory=False) 
                for f in files], ignore_index=True)


 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


In [5]:
print("\n--- Shape ---")
print(df.shape)


--- Shape ---
(2830743, 79)


In [6]:
print("\n--- Class Distribution ---")
print(df[' Label'].value_counts())


--- Class Distribution ---
 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [7]:
print("\n--- Missing Values ---")
print(df.isnull().sum()[df.isnull().sum() > 0])


--- Missing Values ---
Flow Bytes/s    1358
dtype: int64


In [8]:
print("\n--- Data Types ---")
print(df.dtypes.value_counts())


--- Data Types ---
int64      54
float64    24
object      1
Name: count, dtype: int64


In [9]:
desc = df.describe().T 
desc['dtype'] = df.dtypes
print(desc[['dtype', 'count', 'mean', 'std', 'min', 'max']].to_string())

c:\Users\ratch\anaconda3\envs\ds311\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\ratch\anaconda3\envs\ds311\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


                                dtype      count          mean           std           min           max
 Destination Port               int64  2830743.0  8.071483e+03  1.828363e+04  0.000000e+00  6.553500e+04
 Flow Duration                  int64  2830743.0  1.478566e+07  3.365374e+07 -1.300000e+01  1.200000e+08
 Total Fwd Packets              int64  2830743.0  9.361160e+00  7.496728e+02  1.000000e+00  2.197590e+05
 Total Backward Packets         int64  2830743.0  1.039377e+01  9.973883e+02  0.000000e+00  2.919220e+05
Total Length of Fwd Packets     int64  2830743.0  5.493024e+02  9.993589e+03  0.000000e+00  1.290000e+07
 Total Length of Bwd Packets    int64  2830743.0  1.616264e+04  2.263088e+06  0.000000e+00  6.554530e+08
 Fwd Packet Length Max          int64  2830743.0  2.075999e+02  7.171848e+02  0.000000e+00  2.482000e+04
 Fwd Packet Length Min          int64  2830743.0  1.871366e+01  6.033935e+01  0.000000e+00  2.325000e+03
 Fwd Packet Length Mean       float64  2830743.0  5.820

In [10]:
print(desc[['dtype', 'count', 'mean', 'std', 'min', 'max']].head(10).to_string())

                                dtype      count          mean           std   min           max
 Destination Port               int64  2830743.0  8.071483e+03  1.828363e+04   0.0  6.553500e+04
 Flow Duration                  int64  2830743.0  1.478566e+07  3.365374e+07 -13.0  1.200000e+08
 Total Fwd Packets              int64  2830743.0  9.361160e+00  7.496728e+02   1.0  2.197590e+05
 Total Backward Packets         int64  2830743.0  1.039377e+01  9.973883e+02   0.0  2.919220e+05
Total Length of Fwd Packets     int64  2830743.0  5.493024e+02  9.993589e+03   0.0  1.290000e+07
 Total Length of Bwd Packets    int64  2830743.0  1.616264e+04  2.263088e+06   0.0  6.554530e+08
 Fwd Packet Length Max          int64  2830743.0  2.075999e+02  7.171848e+02   0.0  2.482000e+04
 Fwd Packet Length Min          int64  2830743.0  1.871366e+01  6.033935e+01   0.0  2.325000e+03
 Fwd Packet Length Mean       float64  2830743.0  5.820194e+01  1.860912e+02   0.0  5.940857e+03
 Fwd Packet Length Std        

In [11]:
desc[['dtype', 'count', 'mean', 'std', 'min', 'max']].to_csv('descriptive_stats.csv')
print("Saved!")

Saved!


In [12]:
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0])

=== Missing Values ===
Flow Bytes/s    1358
dtype: int64


In [13]:
import numpy as np

print("=== Infinite Values ===")
inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum()
print(inf_count[inf_count > 0])

=== Infinite Values ===
Flow Bytes/s       1509
 Flow Packets/s    2867
dtype: int64


In [14]:
print("=== Outliers (IQR) ===")
outlier_summary = []
for col in df.select_dtypes(include=[np.number]).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    n = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    if n > 0:
        outlier_summary.append({'column': col, 'outlier_%': round(n/len(df)*100, 1)})

result = pd.DataFrame(outlier_summary).sort_values('outlier_%', ascending=False)
print(result.head(10).to_string())

=== Outliers (IQR) ===
                     column  outlier_%
21             Fwd IAT Mean       23.7
9     Fwd Packet Length Std       23.5
6     Fwd Packet Length Max       23.5
20            Fwd IAT Total       23.5
23              Fwd IAT Max       23.5
22              Fwd IAT Std       23.3
13    Bwd Packet Length Std       23.1
40   Packet Length Variance       23.1
10    Bwd Packet Length Max       22.5
0          Destination Port       22.2
